In [1]:
# %%
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Qwen2TokenizerFast
from datasets import load_dataset
from transformers import Qwen3ForCausalLM

from torch.utils.data import DataLoader

model_name = "Qwen/Qwen3-1.7B"
dataset_name = "trl-lib/tldr"


tokenizer: Qwen2TokenizerFast = AutoTokenizer.from_pretrained(model_name)

# Load model with explicit memory management to avoid conflicts with vLLM
train_model: Qwen3ForCausalLM = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},  # Explicitly use GPU 0 instead of "auto"
)

train_model.config.use_cache = False

# Enable gradient checkpointing to save memory
train_model.gradient_checkpointing_enable()

dataset = load_dataset(dataset_name)



dataset = load_dataset(dataset_name)
train_loader = DataLoader(dataset["train"], batch_size=10)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
tokenizer.eos_token

'<|im_end|>'

In [ ]:
prompt = "Give me a short introduction to large language model."
messages = [
    [
    {"role": "user", "content": prompt}
],
[
    {"role": "user", "content": "Give me a short introduction to quantum mechanics as per einstein's reports."}
]
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)


model_inputs = tokenizer(text, return_tensors="pt", padding=True,
    truncation=True, padding_side="left").to(train_model.device)



In [29]:
outputs = train_model.generate(**model_inputs)

In [2]:
import gc
import time
import torch
from vllm import LLM, SamplingParams, EngineArgs, RequestOutput
from vllm.sequence import PromptLogprobs, SampleLogprobs
from transformers import AutoModelForCausalLM, AutoTokenizer, Qwen2TokenizerFast
from datasets import load_dataset
from torch.utils.data import DataLoader
from pydantic import BaseModel
from typing import Optional, Literal, Callable
import wandb as trackio
from tqdm import tqdm
import numpy as np
from transformers import Qwen3ForCausalLM

from torch.optim import Optimizer


def tensor_size_mb(t: torch.Tensor) -> float:
    if not torch.is_tensor(t):
        raise TypeError("tensor_size_mb expects a torch.Tensor")
    return t.numel() * t.element_size() / (1024**2)


def optimizer_state_size_mb(optimizer: Optimizer) -> float:
    def _bytes(obj) -> int:
        if torch.is_tensor(obj):
            return obj.numel() * obj.element_size()
        if isinstance(obj, dict):
            return sum(_bytes(v) for v in obj.values())
        if isinstance(obj, (list, tuple, set)):
            return sum(_bytes(v) for v in obj)
        return 0

    total_bytes = 0
    for state in optimizer.state.values():
        total_bytes += _bytes(state)
    return f"{total_bytes / (1024**3):.4f}GB"


def gpu_mem_allocated(prefix=""):
    print(
        f"{prefix + ' ' if prefix else ''}GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f}GB"
    )


def clear_gpu():
    for r in range(3):
        # Force cleanup
        gc.collect()

    torch.cuda.synchronize()
    torch.cuda.empty_cache()

    with torch.no_grad():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()

    gpu_mem_allocated()


model_name = "Qwen/Qwen3-1.7B"
dataset_name = "trl-lib/tldr"

# Use lower GPU memory for vLLM to leave room for training model
engine_args = EngineArgs(
    model=model_name,
    gpu_memory_utilization=0.15,  # Match the working script
    dtype="bfloat16",
    max_model_len=1024,  # Limit context to save memory
    max_num_batched_tokens=4096,
    max_num_seqs=8,
)

# vllm = LLM(**vars(engine_args))

tokenizer: Qwen2TokenizerFast = AutoTokenizer.from_pretrained(model_name)

dataset = load_dataset(dataset_name)
train_loader = DataLoader(dataset["train"], batch_size=10)


INFO 08-12 02:00:05 [__init__.py:235] Automatically detected platform cuda.


In [28]:
pad_token_id = tokenizer.pad_token_id
eos_token_id = tokenizer.eos_token_id

step = 0
epochs = 10
n_samples = 4
lr = 0.001
batch_size = 8
gradient_accumulation_steps = 4
max_prompt_length = 512
max_completion_lenght = 512

sampling_params = SamplingParams(
    n=n_samples,
    temperature=0.6,
    max_tokens=max_completion_lenght,
    prompt_logprobs=1,
    logprobs=1,
    include_stop_str_in_output=True,
    stop_token_ids=[eos_token_id],
)


template_tokenized = tokenizer.apply_chat_template(
        [
            {
                "role": "system",
                "content": "Your role is to summarize the text into a concise summary. Do not think excessively, focus on summarizing the text.",
            },
            {"role": "user", "content": "The text you must summarize is:\n"},
        ],
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=True,
    )

# convert to chat template
def convert_prompt_to_chat_format(batch: dict) -> list[dict[str, str]]:
    """
    Convert a prompt to a chat format.
    """

    # do stupid things just to clip the right part of the prompt
    prompts_tokenized = tokenizer(
            batch["prompt"],
            truncation=True,
            max_length=(max_prompt_length - len(template_tokenized)),
        )["input_ids"]

    prompts_clipped_decoded = tokenizer.batch_decode(prompts_tokenized)

    return {
        "prompt": tokenizer.apply_chat_template(
            [
                [
                    {
                        "role": "system",
                        "content": "Your role is to summarize the text into a concise summary. Do not think excessively, focus on summarizing the text.",
                    },
                    {
                        "role": "user",
                        "content": "The text you must summarize is:\n"
                        + p.rstrip("TL;DR:").strip(),
                    },
                ]
                for p in prompts_clipped_decoded
            ],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
    }


ds = dataset["train"].map(
    convert_prompt_to_chat_format, remove_columns="completion", batched=True
)

train_loader = DataLoader(ds, batch_size=10)


Map:   0%|          | 0/116722 [00:00<?, ? examples/s]

In [ ]:
for b in train_loader:
    print(b["prompt"][0])
    break

<|im_start|>system
Your role is to summarize the text into a concise summary. Do not think excessively, focus on summarizing the text.<|im_end|>
<|im_start|>user
The text you must summarize is:
SUBREDDIT: r/loseit

TITLE: SV & NSV! Keeping on keeping on.

POST: 30F, 5'6". SW: 236 GW: 150 CW: 219

I weigh myself weekly and measure myself monthly. I'd hit a plateau the last four weeks or so where I was stuck at 222. Felt like kind of a bummer, but knew it's because I haven't been as strict as I should with my diet, and the last week and a half have been crazy with life things, so I haven't been exercising as frequently as I've gotten used to. When I weighed myself as normal on Monday, I was kind of disappointed to see the scale not budging and figured it was time to buckle down again and really watch my diet. Today was my measure-in day, and I've felt cruddy in general since Monday because I caught some chest congestion/cold bug over the weekend. I get on the scale...it says 219. Whaaaaa

In [8]:
print(tokenizer.decode(template_tokenized[-100:]))

<|im_start|>system
Your role is to summarize the text into a concise summary. Do not think excessively, focus on summarizing the text.<|im_end|>
<|im_start|>user
The text you must summarize is:
<|im_end|>



In [ ]:
train_loader = DataLoader(dataset["train"], batch_size=10)

for batch in train_loader:
    



TypeError: argument 'ids': 'list' object cannot be interpreted as an integer

'SUBREDDIT: r/relationships\n\nTITLE: My[25m] girlfriend [24f] is only nice and pleasant when I\'m aloof and distant. (9 months)\n\nPOST: Throwaway\n\nI noticed the more I\'m cold and distant towards my girlfriend, the more pleasant she becomes. She\'ll come over and clean my apartment, do laundry, dishes and cook for me, even as far as to offer oral favors while I\'m drinking a beer! \n\nShe seems completely happy and content during this time, which makes me happy and I naturally want to do things back for her. As soon as I start doing her favors, she picks fights and complains nonstop. Latest issue was I offered to take her and her mom to dinner. She kept giving me shit about how I\'m going to be spending too much time with my brother (who\'s visiting for a week soon), which she was totally fine with when I was being distant with her. She\'ll call me a bitch in a joking way, and just take the piss out of me whenever I\'m kind or go out of my way to apologize. \n\nThis naturally makes